# Context-I2W 加载、单查询及完整验证
运行状态：待 Kaggle 实测。先在右侧 Upload 上传 context_i2w_assets.zip；选择 GPU，开启 Internet。代码来自团队 GitHub 分支。

In [ ]:
import sys, subprocess, os
from pathlib import Path
repo=Path('/kaggle/working/team-baseline')
if not repo.exists():
    subprocess.run(['git','clone','--branch','member-b/baseline-kaggle','https://github.com/Ariel-tian95/Composed-Image-Retrieval.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','ftfy','regex','einops','einops-exts'],check=True)
import torch
assert torch.cuda.is_available(), '请开启 GPU'
print(torch.__version__,torch.cuda.get_device_name(0))

In [ ]:
import zipfile
archives=list(Path('/kaggle/input').rglob('context_i2w_assets.zip'))
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            target=(repo/item.filename).resolve()
            assert target.is_relative_to(repo.resolve())
        z.extractall(repo)
else:
    weights=list(Path('/kaggle/input').rglob('official_checkpoint.pt'))
    images=list(Path('/kaggle/input').rglob('images'))
    assert weights and images, '请上传 assets.zip；若 Kaggle 已自动解压，请确认数据集包含 data/fashion_iq 和 checkpoints'
    source=next(p.parents[2] for p in weights if p.parent.name=='context_i2w')
    import shutil
    for name in ['data','checkpoints']:
        shutil.copytree(source/name,repo/name,dirs_exist_ok=True)
subprocess.run(['git','clone','--depth','1','https://github.com/Pter61/context-i2w.git',str(repo/'external/context_i2w')],check=True) if not (repo/'external/context_i2w').exists() else None
print('图片数：',len(list((repo/'data/fashion_iq/images').glob('*.png'))))
subprocess.run(["git","-C",str(repo/"external/context_i2w"),"fetch","origin","c524d5b9f1551ef14e26791b2bb21654319d0493"],check=True)
subprocess.run(["git","-C",str(repo/"external/context_i2w"),"checkout","c524d5b9f1551ef14e26791b2bb21654319d0493"],check=True)
assert len(list((repo/"data/fashion_iq/images").glob("*.png")))==15415


## 单条查询测试
只使用小图库，输出仅用于检查模型链路，不能作为论文指标。

In [ ]:
subprocess.run([sys.executable,'scripts/run_context_baseline.py','--root',str(repo),'--category','dress','--queries','1','--gallery-limit','64'],check=True)

## 完整验证集评估
仅在单条测试成功后运行。每类分别保存结果；若显存不足，将 batch-size 从16改成8或4。

In [ ]:
for category in ['dress','shirt','toptee']:
    subprocess.run([sys.executable,'scripts/run_context_baseline.py','--root',str(repo),'--category',category,'--queries','0','--batch-size','16'],check=True)

In [ ]:
import json
for p in (repo/'results/baseline').glob('*full.json'):
    d=json.loads(p.read_text());print(d['category'],d['queries'],d['R@10'],d['R@50'])

运行完成后 Save Version，下载 results/baseline 中 JSON。出现错误时保留完整报错，停止后续步骤；本 Notebook 不伪造成功输出。